# **Final Project**


## **Sales Performance Analysis and Profit Optimization of Nike Using Data Analytics**

**Objective**

  The objective of this project is to analyze retail and online sales data of Nike using the Nike Sales (Uncleaned) dataset from Kaggle. The project aims to clean and preprocess raw, inconsistent data using Python to ensure accuracy and reliability. Exploratory Data Analysis (EDA) is performed to identify trends in sales, profit, customer segments, and regional performance. Visualizations and dashboards are created to compare product categories, discounts, and sales channels. The overall goal is to generate meaningful insights that support better business decisions and improve profitability.

**Dataset Source**

Kaggle — Nike Sales (Uncleaned Dataset)

**Dataset link**: https://www.kaggle.com/datasets/nayakganesh007/nike-sales-uncleaned-dataset


# **Stage 1: Problem Definition & Dataset Selection**

**Problem Statement**

The objective of this project is to analyze retail and online sales transaction data to evaluate sales performance, profitability, customer segments, and regional trends. The aim is to clean raw business data, perform exploratory analysis, and generate actionable insights that can help improve sales strategy and operational efficiency.

**The project focuses on identifying:**

*   High-performing regions and product line
*   Revenue and profit trends over timeList item
*   Discount impact on profitability
*   Online vs Retail channel comparison
*   Customer purchasing patterns

**Data selection**
The dataset used is the Nike Sales (Uncleaned) Dataset sourced from Kaggle.
Dataset Characteristics

*   **Records**: 2,500+ transactions
*   **Features**: 12+ column
*   **Data types**: Numerical, categorical, date

**Why this dataset was chosen**

✔ Real-world business scenario

✔ Contains messy data (good for cleaning practice)

✔ Large enough for meaningful analysis

✔ Suitable for dashboard creation

✔ Meets project requirements (500+ rows, 10+ features)


**Data Import in Python**

In [9]:
pip install kaggle

In [10]:
from google.colab import files
files.upload()


Saving kaggle.json to kaggle (1).json


{'kaggle (1).json': b'{"username":"kaviyaanand","key":"f965522a9e6c617fde810b7efcc44521"}'}

In [ ]:
# Set Permission
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d nayakganesh007/nike-sales-uncleaned-dataset

In [ ]:

!unzip nike-sales-uncleaned-dataset.zip

# **Stage 2: Data Cleaning & Pre-processing**

**Cleaning performed using Python in Google Colab**

**Steps implemented:**

**1. Handling Missing Values**

Missing or null values were identified and treated to avoid calculation errors and biased results.

**Actions Taken:**

•	Used .isnull().sum() to detect null values in each column

•	Replaced missing numeric values with median/mean

•	Filled missing categorical values with mode or “Unknown”

Purpose:

•	Prevents errors in aggregation and calculations

•	Maintains dataset completeness



**2. Removing Duplicates**

Duplicate records may lead to inflated sales or revenue figures.

**Actions Taken:**

•	Checked duplicate rows using .duplicated()

•	Removed duplicate Order IDs using .drop_duplicates()

Purpose:

•	Ensures each transaction is counted only once

•	Improves accuracy of KPIs



**3. Correcting Data Types**

Incorrect data types can cause computation and sorting issues.

Actions Taken:
**bold text**
•	Converted Order_Date to datetime format

•	Converted Units_Sold, MRP, Revenue, Profit, and Discount to numeric types (int/float)

Purpose:

•	Enables time-series analysis

•	Supports mathematical operations

Profit Correction

•	Removed negative profit values using absolute()

•	Missing profits estimated as 20% of revenue


**4. Fixing Logical Errors**

Certain values violated real-world business logic.

**Actions Taken:**

•	Removed negative Units_Sold using absolute values

•	Ensured MRP and price values are positive

•	Converted decimal discounts (0.30 → 30%)

Discount(%)=Discount×100

•	Capped discounts above 100%

•	Recalculated Revenue using:

Final Price=MRP×(1−Discount/100)

Revenue=Units Sold×Final Price

Purpose:

•	Ensures realistic and meaningful values

•	Avoids incorrect revenue/profit calculations



**5. Standardizing Text Values**

Inconsistent text formatting creates duplicate categories.

**Actions Taken:**

•	Removed extra spaces

•	Converted text to proper case (Title Case)

•	Corrected region spelling errors (e.g., Delhii → Delhi, Hyd → Hyderabad)

•	Standardized size labels and replaced blanks with “Unknown”

Purpose:

•	Improves grouping and filtering

•	Avoids duplicate categories in analysis

**6. Date Cleaning**

Mixed or invalid date formats affect trend analysis.

**Actions Taken:**

•	Converted dates using pd.to_datetime()

•	Filled missing dates using forward/backward fill

Purpose:

•	Enables monthly/yearly trend analysis

•	Maintains chronological consistency


**7. Feature Engineering (Derived Columns)**

Additional columns were created to enhance analytical insights.

**New Features Created:**

•	Final Price – Price after discount


•	Net Sales – Actual sales after discount

•	Profit Margin (%) – Profitability indicator

•	Year – For yearly comparison

•	Month & Month Name – For monthly trends

•	Average Order Value (AOV) – Revenue per order

Purpose:

•	Supports KPI calculation

•	Improves dashboard insights

•	Enables business decision-making.


In [ ]:
import pandas as pd

df = pd.read_csv("Nike_Sales_Uncleaned.csv")
df.head()

In [ ]:
print(df.shape)
print(df.head())
print(df.info())

In [ ]:
print(df.isnull().sum())


In [ ]:
# 1. Strip spaces
# ==================================================
df.columns = df.columns.str.strip()

for col in df.select_dtypes(include="object"):
    df[col] = df[col].astype(str).str.strip()


# ==================================================
# 2. Convert numeric columns
# ==================================================
num_cols = ["Units_Sold","MRP","Discount_Applied","Revenue","Profit"]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")


# 3. Fix Units Sold (no negatives)

df["Units_Sold"] = df["Units_Sold"].abs()
df["Units_Sold"] = df["Units_Sold"].fillna(df["Units_Sold"].median())



# 4. Fix MRP

df["MRP"] = df["MRP"].abs()
df["MRP"] = df["MRP"].fillna(df["MRP"].median())


# 5. Fix Discount
# If 0.30 → treat as 30%
# Clip 0–100

df.loc[df["Discount_Applied"] <= 1, "Discount_Applied"] *= 100
df["Discount_Applied"] = df["Discount_Applied"].clip(0,100)
df["Discount_Applied"] = df["Discount_Applied"].fillna(0)


# ==================================================
# 6. Fix Size
# ==================================================
df["Size"] = df["Size"].replace(["nan","None",""], "Unknown").str.upper()


# ==================================================
# 7. Fix Regions (typos)
# ==================================================
region_map = {
    "Hydrabad":"Hyderabad",
    "Hyd":"Hyderabad",
    "Bengaluru":"Bangalore",
    "Banglore":"Bangalore",
    "Delhii":"Delhi"
}

df["Region"] = df["Region"].replace(region_map).str.title()


# ==================================================
# 8. Standardize text
# ==================================================
text_cols = ["Gender_Category","Product_Line","Product_Name","Sales_Channel"]
for col in text_cols:
    df[col] = df[col].str.title()


# ==================================================
# 9. Fix Dates (mixed formats)
# ==================================================
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")

# fill missing with nearest valid
df["Order_Date"] = df["Order_Date"].fillna(method="ffill").fillna(method="bfill")


# ==================================================
# 10. Recalculate Revenue correctly
# ==================================================
df["Final_Price"] = df["MRP"] * (1 - df["Discount_Applied"]/100)
df["Revenue"] = df["Units_Sold"] * df["Final_Price"]


# ==================================================
# 11. Fix Profit (no negatives)
# ==================================================
df["Profit"] = df["Profit"].abs()
df["Profit"] = df["Profit"].fillna(df["Revenue"] * 0.2)


# ==================================================
# 12. Add useful columns
# ==================================================
df["Year"] = df["Order_Date"].dt.year
df["Month"] = df["Order_Date"].dt.month
df["Month_Name"] = df["Order_Date"].dt.month_name()


print("Rows after:", len(df))   # same rows


# ==================================================
# 13. Save
# ==================================================
df.to_csv("nike_sales_cleaned_safe.csv", index=False)
print("File saved: nike_sales_cleaned_safe.csv")

In [ ]:
import numpy as np
if "Size" in df.columns:
    df["Size"] = df["Size"].astype(str).str.upper().str.strip()

    apparel = ["S","M","L","XL","XXL"]
    shoe = ["6","7","8","9","10","11","12"]

    df["Apparel_Size"] = df["Size"].where(df["Size"].isin(apparel))
    df["Shoe_Size"] = df["Size"].where(df["Size"].isin(shoe))

    df["Size"] = df["Size"].replace("UNKNOWN", np.nan)

**Save Data**

In [ ]:

# ==================================================
# 13. Save
# ==================================================
df.to_csv("nike_sales_cleaned_safe.csv", index=False)
print("File saved: nike_sales_cleaned_safe.csv")

**Data Download**

In [ ]:
from google.colab import files
files.download("nike_sales_cleaned_safe.csv")

# **Stage 3 : EDA & Visualizations in (POWER BI)**

**Power BI Report Link:**
https://drive.google.com/drive/u/0/folders/1-KrFHsk6kxv1gP0s9y2Fe3-BlyMpPfYT


Exploratory Data Analysis was conducted to uncover trends, patterns, and relationships in the sales data.



**Univariate Analysis**

•	Sales distribution

•	Profit distribution

•	Category frequency


**Bivariate Analysis**

•	Sales by region

•	Profit by product line

•	Discount vs profit relationship


**Multivariate Analysis**

•	Region × Channel × Revenue

•	Time-series trend analysis

•	Correlation between numerical features


# **Stage 4: Documentation, Insights & Presentation**

**Link : https://drive.google.com/drive/u/0/folders/1-KrFHsk6kxv1gP0s9y2Fe3-BlyMpPfYT**

**Power BI Video Link: https://drive.google.com/drive/u/0/folders/1-KrFHsk6kxv1gP0s9y2Fe3-BlyMpPfYT**

# **Future Enhancement**

Begin by analyzing overall sales distribution across regions, product categories, and customer segments to identify high-performing markets and core revenue drivers. This helps understand which products and locations contribute most to business growth.

Next, focus on customer purchasing behavior, including units sold, pricing patterns, discounts, and sales channels (Online vs Retail). These analyses reveal that optimized pricing and controlled discounts play a key role in maximizing profitability.

The dashboard should then highlight profit and revenue breakdowns by product line, gender category, and region to detect underperforming segments and improvement opportunities.

Finally, convert insights into action by identifying high-value products, managing excessive discounts, strengthening online sales strategies, and improving inventory planning to boost overall performance and sustainable growth.

# **Conclusion**

Overall, the analysis shows that sales performance is strongly influenced by region, product demand, and discount strategies. Products with balanced pricing and moderate discounts generate higher profits, while excessive discounts reduce margins. Online channels demonstrate faster growth compared to traditional retail stores. By applying these insights, the business can improve revenue, optimize profit, and make data-driven decisions for long-term success.